# Final-position carry-feature discovery and confirmation

This notebook follows the completed TopK retraining without changing the selected SAE. It first repeats the held-out specificity benchmark at the **final token**, matching the position used for SAE training and graph construction. It then ranks positive graph features on a discovery split by the causal contrast `carry-target delta - matched no-carry delta`, freezes the ordering, and evaluates a predeclared top-10 panel on fresh confirmation cases.

The primary result is confirmatory only if the top-10 panel has a negative carry-minus-control mean, a bootstrap 95% interval wholly below zero, and a negative carry-target effect. The cumulative panel sizes and controls are secondary diagnostics; do not select the best-looking confirmation panel after the run.

## 1. Mount Drive and fetch the current repository

In [ ]:
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

import os
import subprocess

repo_url = 'https://github.com/evey-dev/test_run.git'
repo_dir = '/content/test_run'

def run_cmd(command):
    print('$', ' '.join(map(str, command)))
    subprocess.run(command, check=True)

github_ok = False
try:
    checkout = repo_dir
    if os.path.isdir(os.path.join(checkout, '.git')):
        run_cmd(['git', '-C', checkout, 'pull', '--ff-only'])
    else:
        if os.path.exists(checkout) and os.listdir(checkout):
            checkout = '/content/test_run_github'
        if os.path.isdir(os.path.join(checkout, '.git')):
            run_cmd(['git', '-C', checkout, 'pull', '--ff-only'])
        elif os.path.exists(checkout) and os.listdir(checkout):
            raise RuntimeError(f'{checkout} exists but is not a git repository')
        else:
            run_cmd(['git', 'clone', '--depth', '1', repo_url, checkout])
    os.chdir(checkout)
    github_ok = True
    print('Using GitHub checkout:', os.getcwd())
except Exception as exc:
    print('GitHub checkout failed; using Drive project.zip backup.')
    print(repr(exc))

if not github_ok:
    zip_path = '/content/drive/MyDrive/mphil-project/project.zip'
    if not os.path.exists(zip_path):
        raise FileNotFoundError(f'Could not find {zip_path}')
    run_cmd(['unzip', '-q', '-o', zip_path, '-d', '/content/'])
    for candidate in ['/content/test_run', '/content/mphil_project/test_run', '/content']:
        if os.path.isdir(os.path.join(candidate, 'src')) and os.path.isdir(os.path.join(candidate, 'configs')):
            os.chdir(candidate)
            break
    else:
        raise FileNotFoundError('Could not locate the extracted repository root')

print('Current working directory:', os.getcwd())

## 2. Install dependencies and restore the selected TopK artifacts

In [ ]:
!pip install -q --upgrade "transformers>=4.51.0" accelerate matplotlib pandas
!pip install -q -e .
!python data/generate_datasets.py --capitals

from pathlib import Path
required_followup_files = [
    Path('src/math_carry_feature_screen.py'),
    Path('src/plot_math_carry_screen.py'),
]
missing_followup_files = [str(path) for path in required_followup_files if not path.exists()]
if missing_followup_files:
    raise FileNotFoundError(
        'The GitHub checkout does not contain the follow-up code. Push the current repository changes first: ' +
        ', '.join(missing_followup_files)
    )

In [ ]:
from pathlib import Path
import json
import shutil
import sys

LAYERS = [4, 8, 12, 16, 20, 24, 28]
DRIVE_ROOT = Path('/content/drive/MyDrive/mphil-project')
DRIVE_CHECKPOINT = DRIVE_ROOT / 'mechanistic_data' / 'topk_math_retrain' / 'k256'
DRIVE_OLD_OUTPUT = DRIVE_ROOT / 'outputs' / 'topk_math_retrain'
DRIVE_OUTPUT = DRIVE_ROOT / 'outputs' / 'topk_math_followup'
LOCAL_CHECKPOINT = Path('mechanistic_data/sae_checkpoints_math_topk256')
LOCAL_OLD_OUTPUT = Path('outputs/topk_math_retrain')
LOCAL_OUTPUT = Path('outputs/topk_math_followup')
for path in (DRIVE_OUTPUT, LOCAL_CHECKPOINT, LOCAL_OLD_OUTPUT, LOCAL_OUTPUT):
    path.mkdir(parents=True, exist_ok=True)

for layer in LAYERS:
    for name in (f'sae_layer{layer}.pt', f'sae_layer{layer}_metadata.json'):
        source = DRIVE_CHECKPOINT / name
        destination = LOCAL_CHECKPOINT / name
        if not source.exists():
            raise FileNotFoundError(f'Missing selected TopK checkpoint artifact: {source}')
        if not destination.exists():
            shutil.copy2(source, destination)
            print('Restored', destination)

GRAPH_NAME = 'math_topk256_carry_58_83_4v3_graph.json'
GRAPH_PATH = LOCAL_OLD_OUTPUT / GRAPH_NAME
if not GRAPH_PATH.exists():
    graph_source = DRIVE_OLD_OUTPUT / GRAPH_NAME
    if not graph_source.exists():
        raise FileNotFoundError(f'Missing completed TopK graph: {graph_source}')
    shutil.copy2(graph_source, GRAPH_PATH)

ALL_POSITION_NAME = 'math_topk256_heldout_specificity.json'
all_position_path = LOCAL_OLD_OUTPUT / ALL_POSITION_NAME
if not all_position_path.exists() and (DRIVE_OLD_OUTPUT / ALL_POSITION_NAME).exists():
    shutil.copy2(DRIVE_OLD_OUTPUT / ALL_POSITION_NAME, all_position_path)

print('Checkpoint directory:', LOCAL_CHECKPOINT)
print('Graph:', GRAPH_PATH)
print('Persistent follow-up output:', DRIVE_OUTPUT)

## 3. Position-consistent held-out specificity benchmark

This is the direct comparison to the completed all-position result. It does not retrain the SAE or rebuild the graph.

In [ ]:
last_position_path = LOCAL_OUTPUT / 'math_topk256_heldout_specificity_last.json'
run_cmd([
    sys.executable, '-m', 'src.heldout_validation',
    '--math-sae-config', 'configs/sae_math_topk256_config.yaml',
    '--math-graph', str(GRAPH_PATH),
    '--math-cases', '12',
    '--skip-units',
    '--math-specificity-control',
    '--positions', 'last',
    '--output', str(last_position_path),
])
shutil.copy2(last_position_path, DRIVE_OUTPUT / last_position_path.name)

In [ ]:
import pandas as pd

last_result = json.loads(last_position_path.read_text())
rows = {'final token': last_result['math']['specificity_summary']}
if all_position_path.exists():
    rows['all positions (completed run)'] = json.loads(all_position_path.read_text())['math']['specificity_summary']
comparison = pd.DataFrame({
    label: {
        'carry target delta': value['mean_target_delta'],
        'no-carry control delta': value['mean_no_carry_control_delta'],
        'carry minus control': value['mean_paired_difference'],
        'paired CI': value['bootstrap_95_ci_mean_paired_difference'],
    }
    for label, value in rows.items()
}).T
display(comparison)

## 4. Long discovery/confirmation feature screen

The JSON is written directly to Drive after every screened feature and confirmation panel. Rerun this cell after an interruption: the script verifies the protocol and resumes completed work. The 12 previously inspected benchmark pairs are excluded from both new splits.

In [ ]:
drive_screen_path = DRIVE_OUTPUT / 'math_topk256_carry_feature_screen.json'
run_cmd([
    sys.executable, '-m', 'src.math_carry_feature_screen',
    '--sae-config', 'configs/sae_math_topk256_config.yaml',
    '--graph', str(GRAPH_PATH),
    '--positions', 'last',
    '--discovery-cases', '8',
    '--confirmation-cases', '24',
    '--seed', '1787',
    '--exclude-seed', '787',
    '--exclude-cases', '12',
    '--panel-sizes', '1', '3', '5', '10', '20',
    '--primary-panel-size', '10',
    '--random-panels', '5',
    '--output', str(drive_screen_path),
])
local_screen_path = LOCAL_OUTPUT / drive_screen_path.name
shutil.copy2(drive_screen_path, local_screen_path)

In [ ]:
screen = json.loads(local_screen_path.read_text())
primary = screen['confirmation']['primary_result']
print('Status:', screen['status'])
print('Candidate features:', screen['candidate_feature_count'])
print('Primary panel:', primary['panel'])
print('Predeclared success rule met:', primary['supports_carry_selectivity_under_predeclared_rule'])
display(pd.Series(primary['summary']).to_frame('value'))

panel_table = pd.DataFrame([
    {
        'panel': panel['name'],
        'kind': panel['kind'],
        'features': panel['feature_count'],
        'target delta': panel['summary']['mean_target_delta'],
        'control delta': panel['summary']['mean_no_carry_control_delta'],
        'paired difference': panel['summary']['mean_paired_difference'],
        'paired 95% CI': panel['summary']['bootstrap_95_ci_mean_paired_difference'],
    }
    for panel in screen['confirmation']['panels']
])
display(panel_table)

## 5. Generate figures and persist local artifacts

In [ ]:
figure_dir = LOCAL_OUTPUT / 'figures'
run_cmd([
    sys.executable, '-m', 'src.plot_math_carry_screen',
    '--input', str(local_screen_path),
    '--output-dir', str(figure_dir),
])
from IPython.display import Image, display
display(Image(filename=str(figure_dir / 'fig_math_carry_feature_screen.png')))

for source in LOCAL_OUTPUT.rglob('*'):
    if source.is_file():
        destination = DRIVE_OUTPUT / source.relative_to(LOCAL_OUTPUT)
        destination.parent.mkdir(parents=True, exist_ok=True)
        shutil.copy2(source, destination)
        print('Copied', source.relative_to(LOCAL_OUTPUT))

print('Follow-up complete. Do not choose a secondary panel based on confirmation appearance.')

## Interpretation gate

- If the predeclared top-10 rule is met and the effect is not reproduced by matched random panels, the result supports a compact carry-selective SAE feature set and merits one independent replication graph/prompt.
- If final-token effects are weak but all-position effects remain generic, the earlier improvement arose from a position-mismatched intervention and should not be presented as a carry circuit.
- If the primary confirmation rule fails, stop hyperparameter searching. Report that TopK materially improved sparsity and graph-feature effect size, but neither the matched control nor a held-out feature screen isolated carry-specific causality. This is a limitation of this lightweight SAE/graph implementation, not evidence that all SAEs are intrinsically incapable of representing carry.